In [0]:
%run ../00-common/01-env-variables

In [0]:
%run ../00-common/03-silver-helpers


In [0]:
dbutils.widgets.text('batch_id',"")
v_batch_id = dbutils.widgets.get('batch_id')

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.races"
silver_table = f"{catalog_name}.{silver_schema}.races"


In [0]:
races_df = spark.table(bronze_table)
display(races_df)

In [0]:
reaces_url_removed_df = races_df.select('season',
'round' ,'raceName',
'date',
'circuitId',
'ingestion_timestamp',
'source_file')

In [0]:
reaces_df_reneamed = reaces_url_removed_df.withColumnsRenamed(
    {'raceName' : 'race_name',
     'circuitId' : 'circuit_id'
     })

In [0]:
from pyspark.sql.functions import lit
distinct_races = reaces_df_reneamed.dropDuplicates(['season', 'round']).withColumn('batch_id', lit(v_batch_id))



In [0]:
display(distinct_races)

In [0]:
write_to_silver(distinct_races,
    silver_table,
    "t.season = s.season AND t.round = s.round",
    [
        'season',
        'round',
        'race_name',
        'date',
        'circuit_id',
        'source_file',
        'batch_id',
    ],
    v_batch_id
)